# Tutorial Week 5 -  Game Tree Search

We start with defining the abstract class `Game`, for turn-taking *n*-player games. 

We also define `play_game`, which takes a game and a dictionary of  `{player_name: strategy_function}` pairs, and plays out the game, on each turn checking `state.to_move` to see whose turn it is, and then getting the strategy function for that player and applying it to the game and the state to get a move.

**Reading the trace:** Run cells from top to bottom. `TRACE_SEARCH` switches search messages on/off. `SEARCH_TRACE_DEPTH = 1` prints the root (depth 0) and immediate replies (depth 1). This only limits printing: both algorithms still search to terminal states. Increase cautiously because full game-tree traces can be enormous. Each search reports node visits, terminal evaluations, and (for alpha-beta) cutoff events.

Board construction and win checks occur in every simulated move, so comments explain them without printing on every call. `play_game(..., verbose=True)` prints actual moves and boards.


In [1]:
from collections import namedtuple, Counter, defaultdict
import random
import math
import functools
cache = functools.lru_cache(10**6)

TRACE_SEARCH = True  # Set to False to hide search messages and summaries.
SEARCH_TRACE_DEPTH = 1  # Controls printing only, not the algorithm's search depth.


def search_trace(message, depth):
    # Indent recursion levels and avoid printing the entire enormous search tree.
    if TRACE_SEARCH and depth <= SEARCH_TRACE_DEPTH:
        print(f"{'  ' * depth}[depth {depth}] {message}")

In [2]:
class Game:
    """A game is similar to a problem, but it has a terminal test instead of
    a goal test, and a utility for each terminal state. To create a game,
    subclass this class and implement `actions`, `result`, `is_terminal`,
    and `utility`. You will also need to set the .initial attribute to the
    initial state; this can be done in the constructor."""

    def actions(self, state):
        """Return a collection of the allowable moves from this state."""
        raise NotImplementedError

    def result(self, state, move):
        """Return the state that results from making a move from a state."""
        raise NotImplementedError

    def is_terminal(self, state):
        """Return True if this is a final state for the game."""
        return not self.actions(state)

    def utility(self, state, player):
        """Return the value of this final state to player."""
        raise NotImplementedError


def play_game(game, strategies: dict, verbose=False):
    """Play a turn-taking game. `strategies` is a {player_name: function} dict,
    where function(game, state) is used to get the player's move."""
    state = game.initial
    turn = 0
    if verbose:
        # Print the real game's starting position before choosing any moves.
        print("Starting board:")
        print(state)
    while not game.is_terminal(state):
        turn += 1
        player = state.to_move
        if verbose:
            # Identify whose strategy is about to run and its legal options.
            print(f"Turn {turn}: {player} to move; legal moves={sorted(game.actions(state))}")
        move = strategies[player](game, state)
        state = game.result(state, move)
        if verbose:
            # These are actual played moves, rather than hypothetical search moves.
            print('Player', player, 'move:', move)
            print(state)
    if verbose:
        # The game interface gives terminal utility separately for each player.
        print("Game over; utilities:", {p: game.utility(state, p) for p in strategies})
    return state

# Minimax-Based Game Search Algorithms

We will define several game search algorithms. Each takes two inputs, the game we are playing and the current state of the game, and returns a a `(value, move)` pair, where `value` is the utility that the algorithm computes for the player whose turn it is to move, and `move` is the move itself.

First we define `minimax_search`, which exhaustively searches the game tree to find an optimal move (assuming both players play optimally), and `alphabeta_search`, which does the same computation, but prunes parts of the tree that could not possibly have an affect on the optimnal move.  

In [3]:
def minimax_search(game, state):
    """Search game tree to determine best move; return (value, move) pair."""
    player = state.to_move
    stats = {'visited': 0, 'terminal': 0}

    def max_value(state, depth=0):
        stats['visited'] += 1
        search_trace(f"MAX: choose the best score for root player {player}.", depth)
        if game.is_terminal(state):
            stats['terminal'] += 1
            value = game.utility(state, player)
            # Terminal scores always use the ROOT player's perspective.
            search_trace(f"Terminal state: utility for {player} = {value}", depth)
            return value, None
        v, move = -infinity, None
        for a in game.actions(state):
            # Explore the opponent's best reply after this hypothetical move.
            search_trace(f"MAX considering move {a}", depth)
            v2, _ = min_value(game.result(state, a), depth + 1)
            search_trace(f"MAX move {a} returned value {v2}", depth)
            if v2 > v:
                v, move = v2, a
                search_trace(f"MAX new best: move={move}, value={v}", depth)
        search_trace(f"MAX returns value={v}, move={move}", depth)
        return v, move

    def min_value(state, depth=0):
        stats['visited'] += 1
        search_trace(f"MIN: opponent chooses the worst score for {player}.", depth)
        if game.is_terminal(state):
            stats['terminal'] += 1
            value = game.utility(state, player)
            search_trace(f"Terminal state: utility for {player} = {value}", depth)
            return value, None
        v, move = +infinity, None
        for a in game.actions(state):
            # MIN selects the outcome least favorable to the root player.
            search_trace(f"MIN considering move {a}", depth)
            v2, _ = max_value(game.result(state, a), depth + 1)
            search_trace(f"MIN move {a} returned value {v2}", depth)
            if v2 < v:
                v, move = v2, a
                search_trace(f"MIN new worst for {player}: move={move}, value={v}", depth)
        search_trace(f"MIN returns value={v}, move={move}", depth)
        return v, move

    if TRACE_SEARCH:
        # Separate each new strategy call from recursion inside the previous search.
        print(f"\n[minimax] Searching for {player}; win=+1, draw=0, loss=-1")
    result = max_value(state)
    if TRACE_SEARCH:
        # Counts include hidden levels and repeated visits to equivalent positions.
        print(f"[minimax] Best move={result[1]}, value={result[0]}; "
              f"visited={stats['visited']}, terminal evaluations={stats['terminal']}")
    return result


infinity = math.inf


def alphabeta_search(game, state):
    """Search to terminal states using minimax with alpha-beta pruning."""
    player = state.to_move
    stats = {'visited': 0, 'terminal': 0, 'cutoffs': 0}

    def max_value(state, alpha, beta, depth=0):
        stats['visited'] += 1
        search_trace(f"MAX enters: alpha={alpha}, beta={beta}", depth)
        if game.is_terminal(state):
            stats['terminal'] += 1
            value = game.utility(state, player)
            search_trace(f"Terminal state: utility for {player} = {value}", depth)
            return value, None
        v, move = -infinity, None
        for a in game.actions(state):
            search_trace(f"MAX considering move {a}", depth)
            v2, _ = min_value(game.result(state, a), alpha, beta, depth + 1)
            search_trace(f"MAX move {a} returned value {v2}", depth)
            if v2 > v:
                v, move = v2, a
                # Alpha is the best lower bound established for MAX on this path.
                alpha = max(alpha, v)
                search_trace(f"MAX new best: move={move}, value={v}; alpha={alpha}", depth)
            if v >= beta:
                stats['cutoffs'] += 1
                # MIN already has an option no worse than beta; stop this sibling loop.
                search_trace(f"BETA CUTOFF: value {v} >= beta {beta}; stop this loop.", depth)
                return v, move
        search_trace(f"MAX returns value={v}, move={move}", depth)
        return v, move

    def min_value(state, alpha, beta, depth=0):
        stats['visited'] += 1
        search_trace(f"MIN enters: alpha={alpha}, beta={beta}", depth)
        if game.is_terminal(state):
            stats['terminal'] += 1
            value = game.utility(state, player)
            search_trace(f"Terminal state: utility for {player} = {value}", depth)
            return value, None
        v, move = +infinity, None
        for a in game.actions(state):
            search_trace(f"MIN considering move {a}", depth)
            v2, _ = max_value(game.result(state, a), alpha, beta, depth + 1)
            search_trace(f"MIN move {a} returned value {v2}", depth)
            if v2 < v:
                v, move = v2, a
                # Beta is the best upper bound established for MIN on this path.
                beta = min(beta, v)
                search_trace(f"MIN new worst for {player}: move={move}, value={v}; beta={beta}", depth)
            if v <= alpha:
                stats['cutoffs'] += 1
                # MAX already has an option at least as good as alpha; stop this loop.
                search_trace(f"ALPHA CUTOFF: value {v} <= alpha {alpha}; stop this loop.", depth)
                return v, move
        search_trace(f"MIN returns value={v}, move={move}", depth)
        return v, move

    if TRACE_SEARCH:
        # These bounds restrict useful values, not how far ahead the algorithm looks.
        print(f"\n[alpha-beta] Searching for {player}; win=+1, draw=0, loss=-1")
    result = max_value(state, -infinity, +infinity)
    if TRACE_SEARCH:
        # Count cutoff events, not individual skipped nodes (which are never visited).
        print(f"[alpha-beta] Best move={result[1]}, value={result[0]}; "
              f"visited={stats['visited']}, terminal evaluations={stats['terminal']}, "
              f"cutoff events={stats['cutoffs']}")
    return result

# A Simple Game: Tic-Tac-Toe

We have the notion of an abstract game, we have some search functions; now it is time to define a real game; a simple one, tic-tac-toe. Moves are `(x, y)` pairs denoting squares, where `(0, 0)` is the top left, and `(2, 2)` is the bottom right (on a board of size `height=width=3`).

In [4]:
class TicTacToe(Game):
    """Play TicTacToe on an `height` by `width` board, needing `k` in a row to win.
    'X' plays first against 'O'."""

    def __init__(self, height=3, width=3, k=3):
        self.k = k # k in a row
        self.squares = {(x, y) for x in range(width) for y in range(height)}
        self.initial = Board(height=height, width=width, to_move='X', utility=0)

    def actions(self, board):
        """Legal moves are any square not yet taken."""
        return self.squares - set(board)

    def result(self, board, square):
        """Place a marker for current player on square."""
        player = board.to_move
        # Copy the position so hypothetical search branches do not overwrite each other.
        board = board.new({square: player}, to_move=('O' if player == 'X' else 'X'))
        win = k_in_row(board, player, square, self.k)
        # Store X's score; utility(board, player) converts to the requested perspective.
        board.utility = (0 if not win else +1 if player == 'X' else -1)
        return board

    def utility(self, board, player):
        """Return the value to player; 1 for win, -1 for loss, 0 otherwise."""
        return board.utility if player == 'X' else -board.utility

    def is_terminal(self, board):
        """A board is a terminal state if it is won or there are no empty squares."""
        return board.utility != 0 or len(self.squares) == len(board)

    def display(self, board): print(board)


def k_in_row(board, player, square, k):
    """True if player has k pieces in a line through square."""
    def in_row(x, y, dx, dy): return 0 if board[x, y] != player else 1 + in_row(x + dx, y + dy, dx, dy)
    # Count both directions of each line; subtract the center square counted twice.
    return any(in_row(*square, dx, dy) + in_row(*square, -dx, -dy) - 1 >= k
               for (dx, dy) in ((0, 1), (1, 0), (1, 1), (1, -1)))

States in tic-tac-toe (and other games) will be represented as a `Board`, which is a subclass of `defaultdict` that in general will consist of `{(x, y): contents}` pairs, for example `{(0, 0): 'X', (1, 1): 'O'}` might be the state of the board after two moves. Besides the contents of squares, a board also has some attributes: 
- `.to_move` to name the player whose move it is; 
- `.width` and `.height` to give the size of the board (both 3 in tic-tac-toe, but other numbers in related games);
- possibly other attributes, as specified by keywords. 

As a `defaultdict`, the `Board` class has a `__missing__` method, which returns `empty` for squares that have no been assigned but are within the `width` × `height` boundaries, or `off` otherwise. The class has a `__hash__` method, so instances can be stored in hash tables.

In [5]:
class Board(defaultdict):
    """A board has the player to move, a cached utility value,
    and a dict of {(x, y): player} entries, where player is 'X' or 'O'."""
    empty = '.'
    off = '#'

    def __init__(self, width=8, height=8, to_move=None, **kwds):
        self.__dict__.update(width=width, height=height, to_move=to_move, **kwds)

    def new(self, changes: dict, **kwds) -> 'Board':
        "Given a dict of {(x, y): contents} changes, return a new Board with the changes."
        board = Board(width=self.width, height=self.height, **kwds)
        # Copy existing marks, then apply this hypothetical branch's changes.
        board.update(self)
        board.update(changes)
        return board

    def __missing__(self, loc):
        x, y = loc
        if 0 <= x < self.width and 0 <= y < self.height:
            return self.empty
        else:
            return self.off

    def __hash__(self):
        return hash(tuple(sorted(self.items()))) + hash(self.to_move)

    def __repr__(self):
        def row(y): return ' '.join(self[x, y] for x in range(self.width))
        return '\n'.join(map(row, range(self.height))) +  '\n'

# Players

We need an interface for players. I'll represent a player as a `callable` that will be passed two arguments: `(game, state)` and will return a `move`.
The function `player` creates a player out of a search algorithm, but you can create your own players as functions, as is done with `random_player` below:

In [6]:
def random_player(game, state):
    # Choose a legal square without evaluating any future positions.
    move = random.choice(list(game.actions(state)))
    if TRACE_SEARCH:
        print(f"[random player] {state.to_move} selected {move}")
    return move


def player(search_algorithm):
    """Adapt a search returning (value, move) into a strategy returning only move."""
    def strategy(game, state):
        value, move = search_algorithm(game, state)
        if TRACE_SEARCH:
            # This predicts the result under optimal play, rather than immediate reward.
            print(f"[strategy] {state.to_move}: {search_algorithm.__name__} "
                  f"chose {move}, predicted utility={value}")
        return move
    return strategy

# Playing a Game

We're ready to play a game. I'll set up a match between a `random_player` (who chooses randomly from the legal moves) and a `player(alphabeta_search)` (who makes the optimal alpha-beta move; practical for tic-tac-toe, but not for large games). The `player(alphabeta_search)` will never lose, but if `random_player` is lucky, it will be a tie.

In [7]:
# Random X faces optimal O; O should win or draw.
print("Match: random X versus alpha-beta O")
final_state = play_game(TicTacToe(), dict(X=random_player, O=player(alphabeta_search)), verbose=True)
# The returned board uses +1 for X winning, -1 for O winning, and 0 for a draw.
print("Final utility from X's perspective:", final_state.utility)

Match: random X versus alpha-beta O
Starting board:
. . .
. . .
. . .

Turn 1: X to move; legal moves=[(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 0), (2, 1), (2, 2)]
[random player] X selected (2, 0)
Player X move: (2, 0)
. . X
. . .
. . .

Turn 2: O to move; legal moves=[(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 1), (2, 2)]

[alpha-beta] Searching for O; win=+1, draw=0, loss=-1
[depth 0] MAX enters: alpha=-inf, beta=inf
[depth 0] MAX considering move (0, 1)
  [depth 1] MIN enters: alpha=-inf, beta=inf
  [depth 1] MIN considering move (1, 2)
  [depth 1] MIN move (1, 2) returned value 0
  [depth 1] MIN new worst for O: move=(1, 2), value=0; beta=0
  [depth 1] MIN considering move (2, 1)
  [depth 1] MIN move (2, 1) returned value 0
  [depth 1] MIN considering move (0, 0)
  [depth 1] MIN move (0, 0) returned value -1
  [depth 1] MIN new worst for O: move=(0, 0), value=-1; beta=-1
  [depth 1] MIN considering move (1, 1)
  [depth 1] MIN move (1, 1) returned value -1
  [de

The alpha-beta player will never lose, but sometimes the random player can stumble into a draw. When two optimal (alpha-beta or minimax) players compete, it will always be a draw:

In [8]:
# Both players choose optimally, so standard 3x3 tic-tac-toe should end in a draw.
print("Match: alpha-beta X versus minimax O")
optimal_final_state = play_game(
    TicTacToe(),
    dict(X=player(alphabeta_search), O=player(minimax_search)),
    verbose=True
)
print("Final utility from X's perspective:", optimal_final_state.utility)

Match: alpha-beta X versus minimax O
Starting board:
. . .
. . .
. . .

Turn 1: X to move; legal moves=[(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 0), (2, 1), (2, 2)]

[alpha-beta] Searching for X; win=+1, draw=0, loss=-1
[depth 0] MAX enters: alpha=-inf, beta=inf
[depth 0] MAX considering move (0, 1)
  [depth 1] MIN enters: alpha=-inf, beta=inf
  [depth 1] MIN considering move (1, 2)
  [depth 1] MIN move (1, 2) returned value 1
  [depth 1] MIN new worst for X: move=(1, 2), value=1; beta=1
  [depth 1] MIN considering move (2, 1)
  [depth 1] MIN move (2, 1) returned value 0
  [depth 1] MIN new worst for X: move=(2, 1), value=0; beta=0
  [depth 1] MIN considering move (0, 0)
  [depth 1] MIN move (0, 0) returned value 0
  [depth 1] MIN considering move (1, 1)
  [depth 1] MIN move (1, 1) returned value 0
  [depth 1] MIN considering move (2, 0)
  [depth 1] MIN move (2, 0) returned value 1
  [depth 1] MIN considering move (0, 2)
  [depth 1] MIN move (0, 2) returned value 0
  [depth 

  [depth 1] MIN move (0, 1) returned value 1
  [depth 1] MIN new worst for X: move=(0, 1), value=1; beta=1
  [depth 1] MIN considering move (1, 2)
  [depth 1] MIN move (1, 2) returned value 1
  [depth 1] MIN considering move (2, 1)


  [depth 1] MIN move (2, 1) returned value 1
  [depth 1] MIN considering move (0, 0)
  [depth 1] MIN move (0, 0) returned value 1
  [depth 1] MIN considering move (1, 1)
  [depth 1] MIN move (1, 1) returned value 0
  [depth 1] MIN new worst for X: move=(1, 1), value=0; beta=0
  [depth 1] ALPHA CUTOFF: value 0 <= alpha 0; stop this loop.
[depth 0] MAX move (2, 2) returned value 0
[depth 0] MAX considering move (1, 0)
  [depth 1] MIN enters: alpha=0, beta=inf
  [depth 1] MIN considering move (0, 1)
  [depth 1] MIN move (0, 1) returned value 1
  [depth 1] MIN new worst for X: move=(0, 1), value=1; beta=1
  [depth 1] MIN considering move (1, 2)
  [depth 1] MIN move (1, 2) returned value 0
  [depth 1] MIN new worst for X: move=(1, 2), value=0; beta=0
  [depth 1] ALPHA CUTOFF: value 0 <= alpha 0; stop this loop.
[depth 0] MAX move (1, 0) returned value 0
[depth 0] MAX returns value=0, move=(0, 1)
[alpha-beta] Best move=(0, 1), value=0; visited=23178, terminal evaluations=9221, cutoff events=

  [depth 1] MIN move (2, 2) returned value 0
  [depth 1] MIN considering move (1, 0)
  [depth 1] MIN move (1, 0) returned value 0
  [depth 1] MIN returns value=0, move=(1, 1)
[depth 0] MAX move (0, 0) returned value 0
[depth 0] MAX considering move (1, 1)
  [depth 1] MIN: opponent chooses the worst score for O.
  [depth 1] MIN considering move (1, 2)
  [depth 1] MIN move (1, 2) returned value 0
  [depth 1] MIN new worst for O: move=(1, 2), value=0
  [depth 1] MIN considering move (2, 1)


  [depth 1] MIN move (2, 1) returned value 1
  [depth 1] MIN considering move (0, 0)
  [depth 1] MIN move (0, 0) returned value 0
  [depth 1] MIN considering move (2, 0)
  [depth 1] MIN move (2, 0) returned value 0
  [depth 1] MIN considering move (0, 2)
  [depth 1] MIN move (0, 2) returned value 0
  [depth 1] MIN considering move (2, 2)
  [depth 1] MIN move (2, 2) returned value 0
  [depth 1] MIN considering move (1, 0)
  [depth 1] MIN move (1, 0) returned value 0
  [depth 1] MIN returns value=0, move=(1, 2)
[depth 0] MAX move (1, 1) returned value 0
[depth 0] MAX considering move (2, 0)
  [depth 1] MIN: opponent chooses the worst score for O.
  [depth 1] MIN considering move (1, 2)
  [depth 1] MIN move (1, 2) returned value 1
  [depth 1] MIN new worst for O: move=(1, 2), value=1
  [depth 1] MIN considering move (2, 1)
  [depth 1] MIN move (2, 1) returned value 1
  [depth 1] MIN considering move (0, 0)
  [depth 1] MIN move (0, 0) returned value -1
  [depth 1] MIN new worst for O: move

  [depth 1] MIN move (2, 1) returned value 1
  [depth 1] MIN considering move (0, 0)
  [depth 1] MIN move (0, 0) returned value 1
  [depth 1] MIN considering move (1, 1)
  [depth 1] MIN move (1, 1) returned value 0
  [depth 1] MIN new worst for O: move=(1, 1), value=0
  [depth 1] MIN considering move (2, 0)
  [depth 1] MIN move (2, 0) returned value 0
  [depth 1] MIN considering move (0, 2)
  [depth 1] MIN move (0, 2) returned value -1
  [depth 1] MIN new worst for O: move=(0, 2), value=-1
  [depth 1] MIN considering move (1, 0)
  [depth 1] MIN move (1, 0) returned value 1
  [depth 1] MIN returns value=-1, move=(0, 2)
[depth 0] MAX move (2, 2) returned value -1
[depth 0] MAX considering move (1, 0)
  [depth 1] MIN: opponent chooses the worst score for O.
  [depth 1] MIN considering move (1, 2)
  [depth 1] MIN move (1, 2) returned value 0
  [depth 1] MIN new worst for O: move=(1, 2), value=0
  [depth 1] MIN considering move (2, 1)


  [depth 1] MIN move (2, 1) returned value 1
  [depth 1] MIN considering move (0, 0)
  [depth 1] MIN move (0, 0) returned value -1
  [depth 1] MIN new worst for O: move=(0, 0), value=-1
  [depth 1] MIN considering move (1, 1)
  [depth 1] MIN move (1, 1) returned value -1
  [depth 1] MIN considering move (2, 0)
  [depth 1] MIN move (2, 0) returned value 0
  [depth 1] MIN considering move (0, 2)
  [depth 1] MIN move (0, 2) returned value 1
  [depth 1] MIN considering move (2, 2)
  [depth 1] MIN move (2, 2) returned value 0
  [depth 1] MIN returns value=-1, move=(0, 0)
[depth 0] MAX move (1, 0) returned value -1
[depth 0] MAX returns value=0, move=(2, 1)
[minimax] Best move=(2, 1), value=0; visited=63905, terminal evaluations=29592
[strategy] O: minimax_search chose (2, 1), predicted utility=0
Player O move: (2, 1)
. . .
X . O
. . .

Turn 3: X to move; legal moves=[(0, 0), (0, 2), (1, 0), (1, 1), (1, 2), (2, 0), (2, 2)]

[alpha-beta] Searching for X; win=+1, draw=0, loss=-1
[depth 0] MAX 

### Exercise 1. As in previous tutorials, place print statements so that you can trace the operation of the algorithms.

### Exercise 2. Create a command so that the game is played with two random players.

### Exercise 3. Can you change the game to use a 4x4 board. 
What is the difference in performance. Can you estimate how large the game tree grows in the worst case?
